In [ ]:
from google.colab import files
uploaded = files.upload()  # Select and upload `ml-100k.zip`

Saving u5.base to u5.base
Saving u5.test to u5.test
Saving ua.base to ua.base
Saving ua.test to ua.test
Saving ub.base to ub.base
Saving ub.test to ub.test
Saving u.item to u.item
Saving u.occupation to u.occupation
Saving u.user to u.user
Saving u1.base to u1.base
Saving u1.test to u1.test
Saving u2.base to u2.base
Saving u2.test to u2.test
Saving u3.base to u3.base
Saving u3.test to u3.test
Saving u4.base to u4.base
Saving u4.test to u4.test
Saving allbut.pl to allbut.pl
Saving mku.sh to mku.sh
Saving README to README
Saving u.data to u.data
Saving u.genre to u.genre
Saving u.info to u.info


In [ ]:
import pandas as pd

# Load ratings data
ratings = pd.read_csv("u.data", sep='\t', names=["user_id", "movie_id", "rating", "timestamp"])

# Load movie titles
movies = pd.read_csv("u.item", sep='|', encoding='latin-1', header=None, usecols=[0, 1],
                     names=["movie_id", "title"])

# Merge ratings with movie titles
df = pd.merge(ratings, movies, on="movie_id")

# View the data
df.head()

,user_id,movie_id,rating,timestamp,title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [ ]:
# 1. Restart runtime first (from Runtime > Restart Runtime), then run this:

# 2. Reinstall numpy and surprise in a clean way
!pip uninstall -y numpy
!pip uninstall -y scikit-surprise
!pip install numpy==1.23.5  # Stable version that works with surprise
!pip install scikit-surprise

Found existing installation: numpy 1.23.5
Uninstalling numpy-1.23.5:
  Successfully uninstalled numpy-1.23.5
Found existing installation: scikit-surprise 1.1.4
Uninstalling scikit-surprise-1.1.4:
  Successfully uninstalled scikit-surprise-1.1.4
  Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.3 kB)
Using cached numpy-1.23.5-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.1 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
db-dtypes 1.4.3 requires numpy>=1.24.0, but you have numpy 1.23.5 which is incompatible.
blosc2 3.3.4 requires numpy>=1.26, but you have numpy 1.23.5 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.23.5 which is incompatible.
treescope 0.1.9 requires numpy>=1.2

  Using cached scikit_surprise-1.1.4-cp311-cp311-linux_x86_64.whl


In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# Define rating scale and format for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[["user_id", "movie_id", "rating"]], reader)

# Split the dataset
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Use the SVD model
model = SVD()
model.fit(trainset)

# Make predictions
predictions = model.test(testset)

# Evaluate performance
accuracy.rmse(predictions)

RMSE: 0.9371


0.9370979295041502

In [ ]:
from collections import defaultdict

def get_top_n(predictions, n=5):
    """Return the top-N recommendation for each user from a set of predictions."""
    top_n = defaultdict(list)
    for uid, iid, _, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

# Get top 5 movies recommended for all users
top_n = get_top_n(predictions, n=5)

In [ ]:
user_id = '1'  # must be a string
user_recs = top_n[user_id]
print(f"Top 5 recommendations for User {user_id}:\n")
for movie_id, score in user_recs:
    title = movies[movies["movie_id"] == int(movie_id)]["title"].values[0]
    print(f"{title} (Predicted rating: {score:.2f})")

Top 5 recommendations for User 1:



In [ ]:
print("Available users with predictions:")
print(list(top_n.keys())[:10])  # show the first 10 user IDs

Available users with predictions:
[907, 371, 218, 829, 733, 363, 193, 808, 557, 774]


In [ ]:
user_id = list(top_n.keys())[0]  # choose a real user

In [ ]:
print(len(top_n))  # should be > 0

941


In [ ]:
user_recs = top_n[user_id]

print(f"\nTop 5 movie recommendations for User {user_id}:\n")
for movie_id, score in user_recs:
    title = movies[movies["movie_id"] == int(movie_id)]["title"].values[0]
    print(f"{title} (Predicted rating: {score:.2f})")


Top 5 movie recommendations for User 907:

Princess Bride, The (1987) (Predicted rating: 5.00)
Fugitive, The (1993) (Predicted rating: 5.00)
Psycho (1960) (Predicted rating: 5.00)
Silence of the Lambs, The (1991) (Predicted rating: 5.00)
Toy Story (1995) (Predicted rating: 4.99)


In [ ]:
# Pick a valid user from predictions
user_id = list(top_n.keys())[0]  # pick the first one

user_recs = top_n[user_id]

print(f"\nTop 5 movie recommendations for User {user_id}:\n")
for movie_id, score in user_recs:
    title = movies[movies["movie_id"] == int(movie_id)]["title"].values[0]
    print(f"{title} (Predicted rating: {score:.2f})")


Top 5 movie recommendations for User 907:

Princess Bride, The (1987) (Predicted rating: 5.00)
Fugitive, The (1993) (Predicted rating: 5.00)
Psycho (1960) (Predicted rating: 5.00)
Silence of the Lambs, The (1991) (Predicted rating: 5.00)
Toy Story (1995) (Predicted rating: 4.99)


In [ ]:
from surprise.model_selection import GridSearchCV

# Define the parameter grid to search
param_grid = {'n_epochs': [20, 30], 'lr_all': [0.002, 0.005],
              'reg_all': [0.4, 0.6]}

# Perform GridSearchCV
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=5)
gs.fit(data)

# Print the best RMSE score and parameters
print(f"Best RMSE score: {gs.best_score['rmse']}")
print(f"Best parameters: {gs.best_params['rmse']}")

Best RMSE score: 0.9550646168176422
Best parameters: {'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.4}


In [ ]:
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy

# Define rating scale and format for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[["user_id", "movie_id", "rating"]], reader)

# Train the SVD model with the best parameters on the full dataset
best_params = gs.best_params['rmse']
model = SVD(n_epochs=best_params['n_epochs'],
            lr_all=best_params['lr_all'],
            reg_all=best_params['reg_all'])

# We will train on the full dataset for final model
trainset = data.build_full_trainset()
model.fit(trainset)

print("SVD model trained with best parameters.")

SVD model trained with best parameters.


In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict

# Define rating scale and format for Surprise
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[["user_id", "movie_id", "rating"]], reader)

# Split the dataset into training and testing sets
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Use the SVD model with the best parameters
best_params = gs.best_params['rmse']
model = SVD(n_epochs=best_params['n_epochs'],
            lr_all=best_params['lr_all'],
            reg_all=best_params['reg_all'])

# Train the model on the training set
model.fit(trainset)

# Make predictions on the test set
predictions = model.test(testset)

# Evaluate performance on the test set
print("Model evaluation on the test set:")
accuracy.rmse(predictions)
accuracy.mae(predictions)

# Function to get top-N recommendations
def get_top_n(predictions, n=5):
    """Return the top-N recommendation for each user from a set of predictions."""
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

# Get top 5 movies recommended for all users in the test set
top_n_recommendations = get_top_n(predictions, n=5)

# Print recommendations for a few users (e.g., the first 5 users in the test set)
print("\nTop 5 recommendations for a few users in the test set:")
for i, (user_id, user_recs) in enumerate(list(top_n_recommendations.items())[:5]):
    print(f"\nRecommendations for User {user_id}:")
    for movie_id, score in user_recs:
        # Get the movie title from the movies DataFrame
        title = movies[movies["movie_id"] == int(movie_id)]["title"].values[0]
        print(f"- {title} (Predicted rating: {score:.2f})")

Model evaluation on the test set:
RMSE: 0.9570
MAE:  0.7654

Top 5 recommendations for a few users in the test set:

Recommendations for User 907:
- Silence of the Lambs, The (1991) (Predicted rating: 4.84)
- Ran (1985) (Predicted rating: 4.79)
- Empire Strikes Back, The (1980) (Predicted rating: 4.78)
- Princess Bride, The (1987) (Predicted rating: 4.70)
- Psycho (1960) (Predicted rating: 4.69)

Recommendations for User 371:
- Indiana Jones and the Last Crusade (1989) (Predicted rating: 4.09)
- Blues Brothers, The (1980) (Predicted rating: 4.06)
- Brazil (1985) (Predicted rating: 4.06)
- Dances with Wolves (1990) (Predicted rating: 3.94)
- Professional, The (1994) (Predicted rating: 3.92)

Recommendations for User 218:
- Usual Suspects, The (1995) (Predicted rating: 3.99)
- Chinatown (1974) (Predicted rating: 3.83)
- This Is Spinal Tap (1984) (Predicted rating: 3.70)
- Clerks (1994) (Predicted rating: 3.53)
- Swimming with Sharks (1995) (Predicted rating: 3.48)

Recommendations for Us